In [1]:
import pandas as pd
import numpy as np
import os
import requests
from datetime import datetime

pd.set_option('display.max_rows', 500)


In [2]:
# ── 1. PATH CONFIGURATION ───────────────
BASE_DATA_DIR = "../../../data"

RAW_DIR    = os.path.join(BASE_DATA_DIR, "raw")
BRONZE_DIR = os.path.join(BASE_DATA_DIR, "bronze")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(BRONZE_DIR, exist_ok=True)

CSV_URL = "https://www.data.gouv.fr/api/1/datasets/r/f5df602b-3800-44d7-b2df-fa40a0350325"

path_csv_raw         = os.path.join(RAW_DIR, "liste_communes_france.csv")
path_communes_bronze = os.path.join(BRONZE_DIR, "bronze_liste_communes_france.csv")


In [3]:
# ── 2. RAW LAYER: Landing Zone ──────────────────────────────────────────────
# We keep the source file exactly as it is.
if not os.path.exists(path_csv_raw):
    print(f"📥 Downloading source to RAW...")
    resp = requests.get(CSV_URL, timeout=180)
    resp.raise_for_status()
    with open(path_csv_raw, "wb") as f:
        f.write(resp.content)
    print(f"✅ Saved to RAW: {path_csv_raw}")
else:
    print(f"✅ Raw file already exists in {RAW_DIR}")


📥 Downloading source to RAW...
✅ Saved to RAW: ../../../data/raw/liste_communes_france.csv


In [4]:
# ── 3. BRONZE LAYER: Ingestion & Metadata ───────────────────────────────────
# Bronze is for "raw-ish" data in a readable format with audit columns.
print("\n🛠 Processing RAW -> BRONZE...")
df_bronze = pd.read_csv(path_csv_raw, sep=None, engine="python", dtype=str)

# Normalize column names by trimming surrounding spaces
df_bronze.columns = [c.strip() if isinstance(c, str) else c for c in df_bronze.columns]
df_bronze = df_bronze[df_bronze["dep_code"] == "69"]
df_bronze['extraction_source_url'] = CSV_URL
df_bronze['ingestion_timestamp']   = datetime.now().isoformat()
df_bronze['source_file_name']      = os.path.basename(path_csv_raw)

df_bronze.to_csv(path_communes_bronze, index=False, sep=";", encoding="utf-8")

print(f"✅ BRONZE dataset created: {path_communes_bronze}")
print(f"📊 Rows: {len(df_bronze)} | Columns: {len(df_bronze.columns)}")



🛠 Processing RAW -> BRONZE...
✅ BRONZE dataset created: ../../../data/bronze/bronze_liste_communes_france.csv
📊 Rows: 266 | Columns: 50


In [5]:
df_bronze.head

<bound method NDFrame.head of       Unnamed: 0 code_insee                       nom_standard  \
26969      26969      69001                             Affoux   
26970      26970      69002                         Aigueperse   
26971      26971      69003                  Albigny-sur-Saône   
26972      26972      69004                               Alix   
26973      26973      69005                          Ambérieux   
26974      26974      69006                          Amplepuis   
26975      26975      69007                             Ampuis   
26976      26976      69008                               Ancy   
26977      26977      69009                               Anse   
26978      26978      69010                         L'Arbresle   
26979      26979      69012                      Les Ardillats   
26980      26980      69013                              Arnas   
26981      26981      69014                             Aveize   
26982      26982      69016                   